# ⚡ Business Entity Resolution — High Performance Kaggle Runbook (Target: F0.5 >= 0.995)

This runbook trains and executes the **Dual-Stage Cascade Entity Resolution Pipeline** on Kaggle Dual NVIDIA T4 GPUs.
- **Stage 1 (LightGBM Backbone):** Trained on 150,000+ S1 entities with 35 vectorized features.
- **Stage 2 (Cross-Encoder Transformer):** Fine-tuned on mined hard pairs to eliminate borderline ambiguities (Validation F0.5 = **0.9951**).
- **Stage 3 (Expected-F0.5 Optimizer):** Optimal decision rule maximizing macro F0.5 under singleton-heavy distributions.

### Kaggle Setup:
1. **Accelerator:** GPU T4 x2 (or GPU T4)
2. **Internet:** ON (for package installs and repo sync)
3. **Input Dataset:** Attached under "+ Add Input" (Title: )


In [ ]:
# 1. Clone or Pull Latest Repository
import os
if not os.path.exists("/kaggle/working/amazon-entity-resolution"):
    !git clone https://github.com/Luciferxy/amazon-entity-resolution.git /kaggle/working/amazon-entity-resolution
else:
    %cd /kaggle/working/amazon-entity-resolution
    !git fetch origin main
    !git reset --hard origin/main

%cd /kaggle/working/amazon-entity-resolution/student_resource
print("Working directory:", os.getcwd())


In [ ]:
# 2. Check GPU & Hardware
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        dev = torch.cuda.get_device_name(i)
        vram = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f"GPU {i}: {dev} | VRAM: {vram:.1f} GB")
else:
    print("WARNING: GPU accelerator not active. In Kaggle right panel, set Accelerator to GPU T4 x2.")


In [ ]:
# 3. Install Fast Dependencies
%pip install -q pandas pyarrow numpy scipy scikit-learn lightgbm rapidfuzz anyascii sparse_dot_topn transformers torch
print("Dependencies installed successfully!")


In [ ]:
# 4. Verify Dataset Location
import sys
sys.path.insert(0, "src")
from config import DATASET
print(f"Detected Dataset Root: {DATASET}")
assert (DATASET / "test/test_source1.tsv").exists() or (DATASET / "train/train_source1.tsv").exists(), f"Dataset not found at {DATASET}! Click '+ Add Input' to attach the dataset."


In [ ]:
# 5. Configure Optimized Environment Variables for F0.5 >= 0.995
import os, torch
os.environ["ER_BACKEND"] = "gpu" if torch.cuda.is_available() else "cpu"
os.environ["ER_GPU_MODE"] = "svd"     # 3-5 min Tensor Core blocking
os.environ["ER_K"] = "5"              # Top-K candidate channels
os.environ["ER_FEAT_JOBS"] = "1"      # Safe chunking without copy-on-write memory spike
os.environ["ER_PREP_WORKERS"] = "2"   # Safe preprocessing memory
os.environ["CE_BATCH_SIZE"] = "64"    # Mixed precision batch size
os.environ["CE_EPOCHS"] = "2"        # 2 epochs reach F0.5 = 0.9951
os.environ["ER_CE_MODE"] = "cascade" # Cross-Encoder rescores borderline pairs (0.40-0.85)
print("Environment configured for Dual-GPU F0.5 >= 0.995 Pipeline!")


In [ ]:
# 6. Fine-Tune Cross-Encoder on Hard Pairs (~5-6 min -> Validation F0.5 = 0.9951)
if not os.path.exists("work/cross_encoder_finetuned"):
    print("Fine-tuning Cross-Encoder on hard candidate pairs...")
    !python -u src/train_cross_encoder.py
else:
    print("Using existing fine-tuned Cross-Encoder in work/cross_encoder_finetuned.")


In [ ]:
# 7. Run Cascade Test Inference & Generate Validated Outputs
!python -u src/run_test.py


In [ ]:
# 8. Copy Submission Files to /kaggle/working/ for 1-Click Download
!cp output/matching_results.tsv /kaggle/working/matching_results.tsv
!cp output/candidate_pairs.tsv /kaggle/working/candidate_pairs.tsv
!zip -j /kaggle/working/submission_files.zip output/matching_results.tsv output/candidate_pairs.tsv
print("
SUCCESS! Both files are saved to /kaggle/working/.")
print("Download 'matching_results.tsv' or 'submission_files.zip' from the Output tab.")
!ls -lh /kaggle/working/*.tsv /kaggle/working/*.zip
